In [0]:

# =========================================================
# PHASE 3: DATA QUALITY AUDIT FRAMEWORK
# =========================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

SILVER_TABLE = (
    "workspace.video_game_sales.silver_video_game_sales"
)

AUDIT_TABLE = (
    "workspace.video_game_sales.data_quality_audit"
)

silver_df = spark.table(SILVER_TABLE)

run_id = str(uuid.uuid4())

# Collect actual metrics
total_records = silver_df.count()

valid_years = silver_df.filter(
    F.col("year_status") == "VALID"
).count()

invalid_years = silver_df.filter(
    F.col("year_status") == "MISSING_OR_INVALID"
).count()

sales_matches = silver_df.filter(
    F.col("sales_reconciliation_status") == "MATCH"
).count()

sales_differences = silver_df.filter(
    F.col("sales_reconciliation_status") == "DIFFERENCE_DETECTED"
).count()

null_years = silver_df.filter(
    F.col("year").isNull()
).count()

# Define audit checks
checks = [
    (
        "SILVER_RECORD_COUNT",
        "16598",
        str(total_records),
        "PASS" if total_records == 16598 else "FAIL"
    ),
    (
        "VALID_YEAR_COUNT",
        "16327",
        str(valid_years),
        "PASS" if valid_years == 16327 else "FAIL"
    ),
    (
        "INVALID_YEAR_COUNT",
        "271",
        str(invalid_years),
        "PASS" if invalid_years == 271 else "FAIL"
    ),
    (
        "SALES_MATCH_COUNT",
        "12087",
        str(sales_matches),
        "PASS" if sales_matches == 12087 else "FAIL"
    ),
    (
        "SALES_DIFFERENCE_COUNT",
        "4511",
        str(sales_differences),
        "PASS" if sales_differences == 4511 else "FAIL"
    ),
    (
        "NULL_YEAR_COUNT",
        "271",
        str(null_years),
        "PASS" if null_years == 271 else "FAIL"
    )
]

# Create audit records
audit_records = [
    (
        run_id,
        "SILVER",
        SILVER_TABLE,
        check_name,
        expected,
        actual,
        status
    )
    for check_name, expected, actual, status in checks
]

audit_df = spark.createDataFrame(
    audit_records,
    [
        "run_id",
        "pipeline_layer",
        "table_name",
        "check_name",
        "expected_value",
        "actual_value",
        "status"
    ]
).withColumn(
    "execution_timestamp",
    F.current_timestamp()
)

# Persist audit history
(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

# Display audit results
print("AUDIT RUN ID:", run_id)

display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_id") == run_id)
    .orderBy("check_name")
)

# Fail pipeline if any validation fails
failed_checks = sum(
    1 for check in checks if check[3] == "FAIL"
)

assert failed_checks == 0, "Data quality validation failed!"

print("DATA QUALITY AUDIT PASSED")


In [0]:
spark.sql("""
DESCRIBE TABLE workspace.video_game_sales.data_quality_audit
""").show(truncate=False)

display(
    spark.table(
        "workspace.video_game_sales.data_quality_audit"
    ).orderBy(
        F.desc("execution_timestamp")
    )
)

In [0]:
spark.sql("""
DESCRIBE TABLE workspace.video_game_sales.data_quality_audit
""").show(truncate=False)

display(
    spark.table(
        "workspace.video_game_sales.data_quality_audit"
    ).orderBy(
        F.desc("execution_timestamp")
    )
)

In [0]:
audit_table = "workspace.video_game_sales.data_quality_audit"

display(
    spark.table(audit_table)
    .orderBy("execution_timestamp", ascending=False)
)

In [0]:
display(
    spark.table(audit_table)
    .groupBy("status")
    .count()
)

In [0]:
audit_df = spark.table("workspace.video_game_sales.data_quality_audit")

display(
    audit_df.groupBy("status", "check_name")
    .count()
    .orderBy("check_name")
)